# TinyGRPO-Math

Rule-based GRPO post-training for GSM8K math reasoning.

This notebook is a runner around `scripts/tinygrpo_math.py`, which is the canonical implementation. Keeping the script as the source of truth avoids notebook/script drift while still letting us run and inspect the experiment from VS Code or Jupyter.

Current features covered here:

- `Qwen/Qwen2.5-1.5B-Instruct` default policy/reference model
- GSM8K train split for training and GSM8K test split for evaluation
- `--train_offset` for non-overlapping train slices
- `--init_model_path` for continuation-style policy initialization
- `--eval_model_path` for saved-checkpoint evaluation
- rule-based rewards for answer correctness, final `Answer: <number>` format, parseability, length, and truncation
- optional `math-verify` correctness checking with parser fallback
- detailed JSONL logs and GPU stats

## Install Dependencies

Run this once in the active environment. On Turing, activate `.venv-tinygrpo` first.

In [ ]:
%pip install -U transformers datasets numpy "math-verify[antlr4_13_2]"


## Check Script And CLI

This verifies that the notebook is using the current script and shows all supported arguments.

In [ ]:
from pathlib import Path
import subprocess
import sys

script_path = Path("scripts/tinygrpo_math.py")
print("script exists:", script_path.exists())
print("script path:", script_path.resolve())

subprocess.run([sys.executable, str(script_path), "--help"], check=True)

## Parser And Reward Smoke Test

This does not load a model. It only checks the reward/extraction logic.

In [ ]:
import importlib
from scripts import tinygrpo_math as tg

importlib.reload(tg)
print("math_verify available:", tg.HAS_MATH_VERIFY)

examples = [
    ("Reasoning\nAnswer: 75", "75"),
    ("Reasoning\nFinal Answer: 75.0", "75"),
    ("Reasoning\nAnswer: $75.", "75"),
    ("Reasoning\n\\boxed{75}", "75"),
    ("Reasoning\n\\boxed{\\frac{5}{6}}", "0.8333333333333333333333333333"),
    ("Reasoning\nAnswer: 74", "75"),
]

for completion, gold in examples:
    out = tg.score_completion_components(completion, gold, max_chars=2000)
    print({
        "completion": completion,
        "gold": gold,
        "pred": out["pred"],
        "correct": out["is_correct"],
        "parseable": out["is_parseable"],
        "format": out["final_answer_format"],
        "reward": out["reward"],
    })

## Baseline Eval

Evaluate the base model on held-out GSM8K test examples. Use a new output directory for every run to avoid partial/stale files.

In [ ]:
%%bash
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/tinygrpo_math.py \
  --mode eval \
  --model_name Qwen/Qwen2.5-1.5B-Instruct \
  --eval_size 200 \
  --output_dir outputs/qwen25_15_answer_base_eval200

## First GRPO Run

Train on GSM8K train rows `0:500`. Each prompt samples `G=4` completions.

In [ ]:
%%bash
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/tinygrpo_math.py \
  --mode train \
  --model_name Qwen/Qwen2.5-1.5B-Instruct \
  --train_offset 0 \
  --train_size 500 \
  --eval_size 200 \
  --max_train_updates 500 \
  --group_size 4 \
  --train_batch_size 2 \
  --learning_rate 5e-6 \
  --beta 0.05 \
  --temperature 0.7 \
  --top_p 0.95 \
  --checkpoint_every 500 \
  --output_dir outputs/qwen25_15_grpo500 \
  --save_model

## Evaluate First GRPO Checkpoint

In [ ]:
%%bash
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/tinygrpo_math.py \
  --mode eval \
  --eval_model_path outputs/qwen25_15_grpo500/final_model \
  --eval_size 200 \
  --output_dir outputs/qwen25_15_grpo500_eval200

## Continuation-Style Training

This initializes the policy from the saved GRPO model, but keeps the KL reference anchored to the original base model.

- `--init_model_path`: policy start point (`pi_new`)
- `--model_name`: KL reference (`pi_ref`)
- `--train_offset 500 --train_size 500`: next GSM8K train slice, rows `500:1000`

This is not exact optimizer-state resume. It starts a fresh optimizer from the saved model weights.

In [ ]:
%%bash
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/tinygrpo_math.py \
  --mode train \
  --model_name Qwen/Qwen2.5-1.5B-Instruct \
  --init_model_path outputs/qwen25_15_grpo500/final_model \
  --train_offset 500 \
  --train_size 500 \
  --eval_size 200 \
  --max_train_updates 500 \
  --group_size 4 \
  --train_batch_size 2 \
  --learning_rate 2e-6 \
  --beta 0.05 \
  --temperature 0.7 \
  --top_p 0.95 \
  --checkpoint_every 500 \
  --output_dir outputs/qwen25_15_grpo500_plus500 \
  --save_model

## Evaluate Continuation Checkpoint

In [ ]:
%%bash
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/tinygrpo_math.py \
  --mode eval \
  --eval_model_path outputs/qwen25_15_grpo500_plus500/final_model \
  --eval_size 200 \
  --output_dir outputs/qwen25_15_grpo500_plus500_eval200

## Summarize Eval Outputs

This checks row counts and key metrics. Run after eval folders exist.

In [ ]:
import json
from pathlib import Path

paths = [
    "outputs/qwen25_15_answer_base_eval200/eval_samples.jsonl",
    "outputs/qwen25_15_grpo500_eval200/eval_samples.jsonl",
    "outputs/qwen25_15_grpo500_plus500_eval200/eval_samples.jsonl",
]

for path in paths:
    p = Path(path)
    if not p.exists():
        print("missing:", path)
        continue
    rows = [json.loads(line) for line in p.open()]
    print("\n" + path)
    print("rows:", len(rows))
    print("correct:", sum(x["is_correct"] for x in rows))
    print("answer_marker:", sum(x["ends_with_answer"] for x in rows))
    print("parseable:", sum(x["is_parseable"] for x in rows))
    print("truncated:", sum(x.get("generation", {}).get("truncated_without_eos", False) for x in rows))
    print("avg_reward:", sum(x["reward"] for x in rows) / len(rows))

## Flip Analysis

Compare base versus a trained checkpoint on the same eval rows.

In [ ]:
import json

base_path = "outputs/qwen25_15_answer_base_eval200/eval_samples.jsonl"
trained_path = "outputs/qwen25_15_grpo500_eval200/eval_samples.jsonl"

base = [json.loads(line) for line in open(base_path)]
trained = [json.loads(line) for line in open(trained_path)]

assert len(base) == len(trained), (len(base), len(trained))

counts = {
    "both_correct": 0,
    "base_correct_trained_wrong": 0,
    "base_wrong_trained_correct": 0,
    "both_wrong": 0,
}

for i, (b, t) in enumerate(zip(base, trained)):
    if b["gold"] != t["gold"]:
        raise RuntimeError(f"gold mismatch at {i}: {b['gold']} vs {t['gold']}")
    if b["is_correct"] and t["is_correct"]:
        counts["both_correct"] += 1
    elif b["is_correct"] and not t["is_correct"]:
        counts["base_correct_trained_wrong"] += 1
    elif not b["is_correct"] and t["is_correct"]:
        counts["base_wrong_trained_correct"] += 1
    else:
        counts["both_wrong"] += 1

print(counts)
print("sum:", sum(counts.values()))
print("net gain:", counts["base_wrong_trained_correct"] - counts["base_correct_trained_wrong"])

## Training Signal Analysis

For GRPO, useful groups are prompts where the sampled completions have mixed correctness.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

path = Path("outputs/qwen25_15_grpo500/reward_debug.jsonl")
if not path.exists():
    print("missing:", path)
else:
    groups = defaultdict(list)
    for line in path.open():
        x = json.loads(line)
        groups[(x["epoch"], x["step"], x["prompt_idx"])].append(x)

    correct_dist = Counter()
    same_reward = 0
    same_correct = 0

    for items in groups.values():
        correct_dist[sum(x["is_correct"] for x in items)] += 1
        rewards = {round(x["reward"], 6) for x in items}
        corrects = {x["is_correct"] for x in items}
        same_reward += len(rewards) == 1
        same_correct += len(corrects) == 1

    print("groups:", len(groups))
    print("correct_count_distribution:", dict(sorted(correct_dist.items())))
    print("same_reward_groups:", same_reward)
    print("same_correctness_groups:", same_correct)
    print("useful_mixed_correctness_groups:", len(groups) - same_correct)

## Train/Test Contamination Check

Training uses GSM8K `train`; evaluation uses GSM8K `test`. This checks exact question-text overlap for the first 1000 train rows and first 200 test rows.

In [ ]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
train_questions = set(x["question"].strip() for x in ds["train"].select(range(0, 1000)))
test_questions = set(x["question"].strip() for x in ds["test"].select(range(200)))
overlap = train_questions & test_questions

print("train questions:", len(train_questions))
print("test questions:", len(test_questions))
print("exact overlap:", len(overlap))
for q in list(overlap)[:5]:
    print("OVERLAP:", q)